# Bioinformatics Workshop - SUSTAIN CDP
### Instructors: James Gillespie and Lucy Dillon
contact: j.gillespie@qub.ac.uk

In this workshop, we will learn how to generate embeddings on DNA data, visualise embedded vector data, and build biologically plausible classifiers

In [ ]:
#Import required librarys for workshop
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from Bio import SeqIO
import umap
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_distances
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import os

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)
# Import the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)

max_length = tokenizer.model_max_length

In [ ]:
#set up genome import
genomes = {
    # "E_coli_E11":
    #     "genomes/Escherichia_coli_strain_E11.fasta"

    # "E_coli_E21":
    #     "genomes/Escherichia_coli_strain_E21.fasta",

    "S_aureus":
        "genomes/Staphylococcus_aureus_LACPHL-SPEC-2025-00335.fasta"

    # "S_aureus2":
    #     "genomes/Staphylococcus_aureus_FT526_174.fasta",

    #  "S_aureus3":
    #      "genomes/Staphylococcus_aureus_LACPHL-SPEC-2025-00347.fasta"

}

In [ ]:
#Import windowing code to generate embedding
from embedding import embed_genome

In [ ]:
#generate embedding
genome_embeddings_s = {}
for name, path in genomes.items():

    print(f"Embedding {name}")

    genome_embeddings_s[name] = embed_genome(
        path,
        tokenizer,
        model
    )

    print(
        name,
        genome_embeddings_s[name].shape
    )


all_embeddings_s = np.vstack(
    list(genome_embeddings_s.values())
)

### Since generating embeddings takes quite some time, we will import a range of pre-generated embeddings prepared for this workshop. The genomes belong to either Pseudomonadota or Bacillota, two distinct Phyla.

In [ ]:
all_embeddings = np.load("all_embeddings_1000.npy")

with open("genomes_embeddings_1000.pkl", "rb") as f:
    genome_embeddings = pickle.load(f)

with open("genomes_dict.pkl", "rb") as f:
    genomes = pickle.load(f)

In [ ]:
#We will take a selection of the genomes for the next step:

keep_species = [
    "Escherichia fergusonii",
    "Shigella flexneri",
    "Salmonella enterica",
    "Klebsiella pneumoniae",
    "Enterobacter",
    "Staphylococcus epidermidis",
    "Staphylococcus saprophyticus",
    "Staphylococcus haemolyticus",
]

filtered_embeddings = {
    name: embedding
    for name, embedding in genome_embeddings.items()
    if any(species in name for species in keep_species)
}

filtered_embed = np.vstack(list(filtered_embeddings.values()))

In [ ]:
#to make our plotting more clean, we will assign the species to one of two groups:

labels = []

for name, embedding in filtered_embeddings.items():

    if "Pseudomonadota" in genomes[name]:
        group = "Pseudomonadota"
    elif "Bacillota" in genomes[name]:
        group = "Bacillota"
    else:
        group = "unknown"

    labels.extend([group] * embedding.shape[0])

labels = np.array(labels)

print(filtered_embed.shape[0], len(labels))

In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

X_all = reducer.fit_transform(filtered_embed)

In [ ]:
plt.figure(figsize=(8,6))

colors = {
    "Pseudomonadota": "tab:blue",
    "Bacillota": "tab:red",
    "outgroups": "tab:green",
    "unknown": "tab:gray"
}

for group in np.unique(labels):

    mask = labels == group

    plt.scatter(
        X_all[mask, 0],
        X_all[mask, 1],
        s=10,
        alpha=0.1,
        color=colors[group],
        label=group
    )

plt.legend(title="Group")
plt.title("Genome Embedding UMAP")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

In [ ]:
#Now we can fit new data to our UMAP
#Lets see where the Staphylococcus genome we generated an embedding for sits
X_staph = reducer.transform(all_embeddings_s)

In [ ]:
plt.figure(figsize=(8,6))

# Existing points
for group in np.unique(labels):

    mask = labels == group

    plt.scatter(
        X_all[mask, 0],
        X_all[mask, 1],
        s=2,
        alpha=0.3,
        label=group
    )

# Overlay Strap
plt.scatter(
    X_staph[:, 0],
    X_staph[:, 1],
    s=10,
    alpha=0.3,
    c="black",
    label="Staph"
)

plt.legend()
plt.show()

In [ ]:
#We can compare our embedding results to a k-mers benchmark
#K-mers works with a sliding window across the genome
#Please open kmers_analysis.py and examine the code.
from kmers_analysis import genome_to_kmers

In [ ]:
filtered_path = {
    name: path
    for name, path in genomes.items()
    if any(species in name for species in keep_species)
}

In [ ]:
genome_kmers = {}

for name, path in filtered_path.items():
    fpath = path.split("/Bioinformatics_Workshop_Gillespie/", 1)[1]
    genome_name = name

    genome_kmers[genome_name] = genome_to_kmers(
        fpath,
        window_size=2000,
        stride=1000,
        k=6
    )

In [ ]:
all_kmers = []
k_labels = []

for genome_name, kmers in genome_kmers.items():

    all_kmers.append(kmers)

    k_labels.extend(
        [genome_name] * len(kmers)
    )

all_kmers = np.vstack(all_kmers)
k_labels = np.array(k_labels)

In [ ]:
#to make our plotting more clean, we will assign the species to one of two groups:

kk_labels = []

for name, kmers in genome_kmers.items():

    if "Pseudomonadota" in genomes[name]:
        group = "Pseudomonadota"
    elif "Bacillota" in genomes[name]:
        group = "Bacillota"
    else:
        group = "unknown"

    kk_labels.extend([group] * len(kmers))

kk_labels = np.array(kk_labels)

In [ ]:
k_reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

X_kmers = k_reducer.fit_transform(all_kmers)

In [ ]:
plt.figure(figsize=(8,6))

for genome in np.unique(kk_labels):

    mask = kk_labels == genome

    plt.scatter(
        X_kmers[mask, 0],
        X_kmers[mask, 1],
        s=5,
        alpha=0.6,
        label=genome
    )

plt.legend()
plt.title("Genome K-mers UMAP")
plt.show()

In [ ]:
#Run k-mers on unknown genome

unknown_genome = genome_to_kmers(
        'genomes/Staphylococcus_aureus_LACPHL-SPEC-2025-00335.fasta',
        window_size=2000,
        stride=1000,
        k=6
    )

In [ ]:
K_straph = k_reducer.transform(unknown_genome)

In [ ]:
plt.figure(figsize=(8,6))

# Existing points
for group in np.unique(kk_labels):

    mask = kk_labels == group

    plt.scatter(
        X_kmers[mask, 0],
        X_kmers[mask, 1],
        s=2,
        alpha=0.3,
        label=group
    )

# Overlay Strap
plt.scatter(
    K_straph[:, 0],
    K_straph[:, 1],
    s=10,
    alpha=0.3,
    c="black",
    label="Staph"
)

plt.legend()
plt.show()

### Can you generate an embedding for E.Coli and plot where it lands on the umap?

In [ ]:
# Uncomment E.Coli genome path
# Rerun embedding code
# Rerun plotting code

# You may wish to do this in the cells below so that you dont lose your Staphylococcus aureus plots

## Can you add a PCA plot to visualise the same data?

------------------------------

## Now we will build some classifiers

In [ ]:
#We can test our classifier on other genomes we have generated. Simply pass the genome name into the predict() function.
#The list of genome names which are pre-computed but not included in the training data are as follows:

for name, n in genomes.items():
    if name not in filtered_embeddings:
        print (name)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
X = []
y = []
groups = []

for name, embedding in filtered_embeddings.items():

    path = genomes[name]

    if "Pseudomonadota" in path:
        label = "Pseudomonadota"
    elif "Bacillota" in path:
        label = "Bacillota"
    else:
        continue

    X.append(embedding)
    y.extend([label] * embedding.shape[0])
    groups.extend([name] * embedding.shape[0])

X = np.vstack(X)
y = np.array(y)
groups = np.array(groups)

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

models = {
    "k-NN": KNeighborsClassifier(metric="cosine"),
    "Logistic": LogisticRegression(max_iter=1000),
    "Linear SVM": SVC(kernel="linear")
}

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print(name)
    print(accuracy_score(y_test, pred))

In [ ]:
from collections import Counter
votes = Counter(models['Logistic'].predict(genome_embeddings['Streptococcus pneumoniae LACPHL']))

print(votes)

### We can calucate the confusion matrix for our trained classifier. Are you able to produce the confusion matricies for the other classifiers?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
pred = models['k-NN'].predict(X_test)

print(accuracy_score(y_test, pred))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred
)

### What genome has the worst classification? Can we think of reasons why it might have classified poorly?

### Can we reproduce the work above but for E.Coli Versus Tuberculosis? What can we learn about the two species from the UMAP / PCA plots?

In [ ]:
ecoli_embeddings = np.load("all_embeddings_ecoli.npy")

with open("genomes_embedding_ecoli.pkl", "rb") as f:
    genome_embeddings_ecoli = pickle.load(f)

with open("genomes_dict_ecoli.pkl", "rb") as f:
    ecoli_genomes = pickle.load(f)

In [ ]:
#generate labels for species as in previously used code

#run umap reducer and plot


# Part X

### Let us demonstrate the difference between shuffling a k-mer and shuffling an embedding

In [ ]:
new_genomes = {
    "Original": "genomes/genomes_original.fasta",
    "Shuffled": "genomes/genomes_shuffled_k6.fasta"
}

In [ ]:
new_genome_embeddings = {}
nt_model = AutoModelForMaskedLM.from_pretrained("InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", trust_remote_code=True)
for name, path in new_genomes.items():

    print(f"Embedding {name}")

    new_genome_embeddings[name] = embed_genome(
        path,
        tokenizer,
        nt_model
    )

    print(
        name,
        new_genome_embeddings[name].shape
    )


new_all_embeddings = np.vstack(
    list(new_genome_embeddings.values())
)

In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

X_shuffled = reducer.fit_transform(new_all_embeddings)

In [ ]:

labels = []

for name, embedding in new_genome_embeddings.items():

    if "original" in new_genomes[name]:
        group = "original"
    elif "shuffled" in new_genomes[name]:
        group = "shuffled"
    else:
        group = "unknown"

    labels.extend([group] * embedding.shape[0])

labels = np.array(labels)

In [ ]:
plt.figure(figsize=(8,6))


for group in np.unique(labels):

    mask = labels == group

    plt.scatter(
        X_shuffled[mask, 0],
        X_shuffled[mask, 1],
        s=10,
        alpha=0.3,
        label=group
    )

plt.legend(title="Group")
plt.title("Shuffled Genome Embedding UMAP")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

In [ ]:
genome_kmers = {}

for name, path in new_genomes.items():
    print(path)
    fpath = path
    genome_name = name

    genome_kmers[genome_name] = genome_to_kmers(
        fpath,
        window_size=2000,
        stride=1000,
        k=6
    )

In [ ]:
all_kmers = []
k_labels = []

for genome_name, kmers in genome_kmers.items():

    all_kmers.append(kmers)

    k_labels.extend(
        [genome_name] * len(kmers)
    )

all_kmers = np.vstack(all_kmers)
k_labels = np.array(k_labels)

kk_labels = []

for name, kmers in genome_kmers.items():

    if "original" in new_genomes[name]:
        group = "original"
    elif "shuffled" in new_genomes[name]:
        group = "shuffled"
    else:
        group = "unknown"

    kk_labels.extend([group] * len(kmers))

kk_labels = np.array(kk_labels)

k_reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

X_kmers = k_reducer.fit_transform(all_kmers)

plt.figure(figsize=(8,6))

for genome in np.unique(kk_labels):

    mask = kk_labels == genome

    plt.scatter(
        X_kmers[mask, 0],
        X_kmers[mask, 1],
        s=5,
        alpha=0.6,
        label=genome
    )

plt.legend()
plt.title("Shuffled Genome K-mers UMAP")
plt.show()

### The UMAP results show that shuffling the genome disrupts the biological signal captured by the embedding space. The k-mer analysis remains largely unchanged because k-mers treat the genome as a bag of substrings; they only capture frequency information, not sequence order. As a result, the overall k-mer distribution is preserved under shuffling, even though the underlying biological structure has been destroyed. This highlights that k-mer methods are insensitive to sequence arrangement, whereas embeddings encode contextual and positional information beyond simple token counts.

### We can also compute metrics which illustrate this in a quantitive way:

In [ ]:
# The silhouette score computes how well-separated clusters are by comparing the average distance 
# between points within the same class to the distance between points in different classes.
# Higher values indicate tighter, more distinct clustering.

from sklearn.metrics import silhouette_score

sil_kmer = silhouette_score(X_kmers, labels, metric="euclidean")
sil_emb = silhouette_score(X_shuffled, labels, metric="cosine")

print("K-mer score: ", sil_kmer, "Embedding score: ", sil_emb)

In [ ]:
# Nearest neighbours computes how often a point’s closest samples belong to the same class, 
# measuring local structure consistency and how biologically coherent the neighbourhood structure is.
# A higher nearest neighbour score means that a point’s closest samples are more likely to belong to the same class or species
import numpy as np
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier

def nn_purity(X, labels, k=5):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine").fit(X)
    _, idx = nn.kneighbors(X)

    scores = []
    for i, neighbors in enumerate(idx):
        scores.append(np.mean(labels[neighbors] == labels[i]))

    return np.mean(scores)

print("K-mer score ", nn_purity(X_kmers, labels))
print("Embedding score ", nn_purity(X_shuffled, labels))

In [ ]:
# The logistic regression classifier shows us how linearly separable the classes are in the feature space, 
# i.e. whether a simple linear decision boundary can distinguish between species or groups based on the representations.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

clf = LogisticRegression(max_iter=2000)

scores = cross_val_score(clf, X_shuffled, labels, cv=5)
print("Embedding score ", scores.mean())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

clf = LogisticRegression(max_iter=2000)

scores = cross_val_score(clf, X_kmers, labels, cv=5)
print("K-mre score ", scores.mean())